Problem Statement:
- For pairs of brands in the same year (e.g. apple/samsung/2020 and samsung/apple/2020) 
      - if custom1 = custom3 and custom2 = custom4 : then keep only one pair
- For pairs of brands in the same year 
      - if custom1 != custom3 OR custom2 != custom4 : then keep both pairs
- For brands that do not have pairs in the same year : keep those rows as well

In [0]:
%sql
DROP TABLE IF EXISTS brands;
CREATE TABLE brands 
(
    brand1      VARCHAR(20),
    brand2      VARCHAR(20),
    year        INT,
    custom1     INT,
    custom2     INT,
    custom3     INT,
    custom4     INT
);
INSERT INTO brands VALUES ('apple', 'samsung', 2020, 1, 2, 1, 2);
INSERT INTO brands VALUES ('samsung', 'apple', 2020, 1, 2, 1, 2);
INSERT INTO brands VALUES ('apple', 'samsung', 2021, 1, 2, 5, 3);
INSERT INTO brands VALUES ('samsung', 'apple', 2021, 5, 3, 1, 2);
INSERT INTO brands VALUES ('google', NULL, 2020, 5, 9, NULL, NULL);
INSERT INTO brands VALUES ('oneplus', 'nothing', 2020, 5, 9, 6, 3);

SELECT * FROM brands;

SQL SOLUTION

In [0]:
%sql
with cte(
select *, case when brand1 < brand2 then concat(brand1, brand2, year)
          else concat(brand2, brand1, year) end as pairid
from brands),
      cte_rnk as (
        select *, row_number() over(partition by pairid order by pairid) as rn
        from cte
      )

select brand1, brand2, year, custom1, custom2, custom3, custom4
from cte_rnk
where rn = 1 or (custom1 <> custom3 and custom2 <> custom4)

PySpark Solution

In [0]:
from pyspark.sql import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import *

# 1. Define the Schema (matches your SQL DDL)
schema = StructType([
    StructField("brand1", StringType(), True),
    StructField("brand2", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("custom1", IntegerType(), True),
    StructField("custom2", IntegerType(), True),
    StructField("custom3", IntegerType(), True),
    StructField("custom4", IntegerType(), True)
])

# 2. Define the Data (matches your INSERT statements)
data = [
    ('apple', 'samsung', 2020, 1, 2, 1, 2),
    ('samsung', 'apple', 2020, 1, 2, 1, 2),
    ('apple', 'samsung', 2021, 1, 2, 5, 3),
    ('samsung', 'apple', 2021, 5, 3, 1, 2),
    ('google', None, 2020, 5, 9, None, None),
    ('oneplus', 'nothing', 2020, 5, 9, 6, 3)
]

# 3. Create the DataFrame
brands_df = spark.createDataFrame(data, schema)

# 4. Display the result
brands_df.display()

In [0]:
brands_df = brands_df.withColumn("pairid", when(col("brand1")<col("brand2"), concat(col("brand1"), col("brand2"), col("year")))
                                .otherwise( concat(col("brand2"), col("brand1"), col("year"))) )

brands_df.display()

window_spec = Window().partitionBy("pairid").orderBy("pairid")
brands_df = brands_df.withColumn("rn", row_number().over(window_spec))
brands_df.display()

brands_df.filter((col("rn")==1) | ((col("custom1")!=col("custom3")) & (col("custom2")!=col("custom4")))).display()